In [3]:
import duckdb

parquet_file = "C:/Workspace/06_ML_projdect/26_1_COIN/data/ST4000DM000_temp2.parquet"

con = duckdb.connect()
df = con.execute(f"""
    SELECT has_failed, COUNT(*) as object_count
    FROM (
        SELECT serial_number, MAX(CAST(failure AS INTEGER)) as has_failed
        FROM read_parquet('{parquet_file}') 
        WHERE smart_241_raw IS NULL
        GROUP BY serial_number
    )
    GROUP BY has_failed
""").fetchdf()
con.close()

print("=== 아예 비어있는 '하드디스크 객체(대수)' 고장 여부 ===")
print("0: 정상(Healthy) 대수 / 1: 고장(Failed) 대수")
print(df)


=== 아예 비어있는 '하드디스크 객체(대수)' 고장 여부 ===
0: 정상(Healthy) 대수 / 1: 고장(Failed) 대수
   has_failed  object_count
0           1            79
1           0          5987


In [4]:
# 특정 센서 누락 객체 랜덤 탐색

import duckdb
import pandas as pd

# 1차 보간(Phase 1.3) 후 결측치가 남은 파일 (복원하신 파일)
parquet_file = "C:/Workspace/06_ML_projdect/26_1_COIN/data/ST4000DM000_temp2.parquet"

# 판다스가 중간 행을 생략(,,,)하지 않고 다 보여주도록 설정
pd.set_option('display.max_rows', 200)

con = duckdb.connect()

# 1. 완전 결측된 90대 하드디스크 중 1대의 serial_number를 가져옵니다.
target_serial = con.execute(f"""
    SELECT serial_number 
    FROM read_parquet('{parquet_file}') 
    WHERE smart_241_raw IS NULL 
    ORDER BY random()
    LIMIT 1
""").fetchone()[0]

print(f"🎯 타겟 하드디스크 시리얼 번호: {target_serial}")

# 2. 해당 하드디스크의 전체 생애(모든 날짜) 기록과 문제의 5개 센서 열을 뽑아봅니다.
df = con.execute(f"""
    SELECT 
        date, 
        failure, 
        smart_241_raw, smart_242_raw, smart_240_raw, smart_190_raw, smart_7_raw
    FROM read_parquet('{parquet_file}')
    WHERE serial_number = '{target_serial}'
    ORDER BY date
""").fetchdf()
con.close()

print(f"\n=== {target_serial} 디스크의 전체 생애 센서 이력 ===")
display(df)  # Jupyter Notebook용 출력


🎯 타겟 하드디스크 시리얼 번호: W300BN40

=== W300BN40 디스크의 전체 생애 센서 이력 ===


,date,failure,smart_241_raw,smart_242_raw,smart_240_raw,smart_190_raw,smart_7_raw
0,2013-06-27,0,None,None,None,None,None
1,2013-06-28,0,None,None,None,None,None
2,2013-06-29,0,None,None,None,None,None
3,2013-06-30,0,None,None,None,None,None
4,2013-07-01,0,None,None,None,None,None
...,...,...,...,...,...,...,...
1817,2018-06-18,0,32951362816,186608148239,41164,32,302134996
1818,2018-06-19,0,32951362816,186608148239,41183,31,302135237
1819,2018-06-20,0,32951362816,186608148239,41201,31,302135471
1820,2018-06-21,0,32951362816,186608148239,41218,31,302135669


In [1]:
import duckdb
from pathlib import Path
from tqdm.auto import tqdm
import gc # 파이썬 메모리 강제 청소기

def convert_csv_to_parquet_duckdb_ultimate():
    src_dir = r"C:\Workspace\06_ML_projdect\26_1_COIN\data\raw_data_csv"
    dest_dir = r"C:\Workspace\06_ML_projdect\26_1_COIN\data\raw_data_parquet"
    
    src_path = Path(src_dir)
    dest_path = Path(dest_dir)
    dest_path.mkdir(parents=True, exist_ok=True)
    
    print("CSV 파일 탐색 및 월별 분류 중...")
    csv_files = list(src_path.rglob("*.csv"))
    
    monthly_files = {}
    for file_path in csv_files:
        month = file_path.stem[:7]
        if month not in monthly_files:
            monthly_files[month] = []
        monthly_files[month].append(str(file_path).replace('\\', '/'))
        
    print(f"총 {len(monthly_files)}개월 치의 데이터를 DuckDB를 이용해 병합합니다.\n")
    
    for month, files in tqdm(monthly_files.items(), desc="DuckDB 병합 (강제 램 관리 모드)"):
        output_filepath = dest_path / f"{month}.parquet"
        
        # 이미 변환된 파케이 파일 건너뛰기
        if output_filepath.exists():
            continue
            
        files.sort()
        output_str = str(output_filepath).replace('\\', '/')
        quoted_files = [f"'{f}'" for f in files]
        files_array_str = "[" + ", ".join(quoted_files) + "]"
        
        con = duckdb.connect()
        
        try:
            # 🌟 [초강력 조치 1] DuckDB가 사용할 수 있는 최대 램 상한선을 8GB로 막아버림!
            # 이렇게 하면 램이 꽉 차기 전에 알아서 내부 데이터를 버리고 비우며 작동합니다.
            con.execute("PRAGMA memory_limit='8GB';")
            
            query = f"""
                COPY (
                    SELECT * FROM read_csv(
                        {files_array_str}, 
                        union_by_name=true, 
                        ignore_errors=true,
                        types={{'date': 'VARCHAR', 'serial_number': 'VARCHAR', 'model': 'VARCHAR'}}
                    )
                ) TO '{output_str}' (FORMAT PARQUET);
            """
            con.execute(query)
            
        except Exception as e:
            print(f"[{month}] 변환 중 오류: {e}")
            
        finally:
            con.close()
            
            # 🌟 [초강력 조치 2] 한 바퀴 돌 때마다 메모리 청소부를 강제로 부름!
            # 이전 찌꺼기를 OS 레벨에서 완전히 즉시 반환하도록 멱살을 잡습니다.
            gc.collect()
            
    print("\n🎉 모든 변환이 안전하게 완료되었습니다!")

# 실행
if __name__ == "__main__":
    convert_csv_to_parquet_duckdb_ultimate()


CSV 파일 탐색 및 월별 분류 중...
총 147개월 치의 데이터를 DuckDB를 이용해 병합합니다.



DuckDB 병합 (강제 램 관리 모드):   0%|          | 0/147 [00:00<?, ?it/s]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


🎉 모든 변환이 안전하게 완료되었습니다!


In [4]:
import duckdb
import pandas as pd

# 1. DuckDB 연결 생성 (메모리 모드)
con = duckdb.connect()

# 2. 전체 Parquet 파일을 읽어서 HDD 모델별 통계를 집계하는 강력한 SQL 쿼리
query = """
SELECT 
    model AS "모델",
    COUNT(*) AS "행 수",
    COUNT(DISTINCT serial_number) AS "개체 수",
    COUNT(DISTINCT CASE WHEN failure = 1 THEN serial_number END) AS "고장 개체 수",
    ROUND(CAST(COUNT(DISTINCT CASE WHEN failure = 1 THEN serial_number END) AS DOUBLE) / 
          CAST(COUNT(DISTINCT serial_number) AS DOUBLE) * 100, 4) AS "고장률 (%)"
FROM read_parquet('C:/Workspace/06_ML_projdect/26_1_COIN/data/raw_data_parquet/*.parquet')
GROUP BY model
ORDER BY "개체 수" DESC
LIMIT 20
"""

print("전체 데이터에서 모델별 통계 집계를 시작합니다. (수십 초가량 소요될 수 있습니다)")

# 3. 쿼리 실행 후 결과를 Pandas DataFrame으로 가져오기
model_stats_df = con.execute(query).fetchdf()

# 4. 결과 출력
display(model_stats_df.head(50))


전체 데이터에서 모델별 통계 집계를 시작합니다. (수십 초가량 소요될 수 있습니다)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,모델,행 수,개체 수,고장 개체 수,고장률 (%)
0,TOSHIBA MG08ACA16TA,27000887,40996,858,2.0929
1,TOSHIBA MG07ACA14TA,64805775,39387,1826,4.6360
2,ST12000NM0007,37055961,38843,2262,5.8234
3,WDC WUH722222ALE6L4,10978372,37451,243,0.6488
4,ST4000DM000,80454439,37040,5790,15.6317
5,ST16000NM001G,34692793,34755,684,1.9681
6,WDC WUH721816ALE6L4,21126913,26597,214,0.8046
7,ST12000NM0008,37903847,21037,2093,9.9491
8,HGST HMS5C4040BLE640,41138908,16349,448,2.7402
9,ST8000NM0055,41512527,15680,2255,14.3814
